In [11]:
import os
import zipfile
import pandas as pd
import geopandas as gpd
from bs4 import BeautifulSoup
from shapely.geometry import Polygon

KMZ → DataFrame

In [12]:
def kmz_to_dataframe(kmz_file_path):
    """
    Extract placemark data from KMZ file into DataFrame.
    Returns DataFrame with columns: Id, x, y, gridcode
    """
    with zipfile.ZipFile(kmz_file_path, 'r') as kmz:
        kml_filename = [f for f in kmz.namelist() if f.endswith('.kml')][0]
        with kmz.open(kml_filename, 'r') as kml_file:
            kml_content = kml_file.read()

    soup = BeautifulSoup(kml_content, 'xml')

    rows = []
    for placemark in soup.find_all('Placemark'):
        coords_tag = placemark.find('coordinates')
        if not coords_tag:
            continue
        
        coords_raw = coords_tag.text.strip().split()[0]
        lon, lat = coords_raw.split(',')[:2]

        rows.append({
            'Id': placemark.find('name').text if placemark.find('name') else None,
            'x': float(lon),
            'y': float(lat),
            'gridcode': _extract_gridcode(placemark.find('description'))
        })

    df = pd.DataFrame(rows)
    return df

def _extract_gridcode(description):
    """Extract gridcode from description element."""
    if not description:
        return None
    
    html_soup = BeautifulSoup(description.text, 'html.parser')
    gridcode = _extract_gridcode_from_table(html_soup)
    return gridcode if gridcode else _extract_gridcode_from_text(html_soup)

def _extract_gridcode_from_table(html_soup):
    """Extract gridcode from HTML table format."""
    for td in html_soup.find_all('td'):
        text = td.get_text(strip=True)
        if text.lower() == 'gridcode':
            next_td = td.find_next_sibling('td')
            if next_td:
                return next_td.get_text(strip=True)
    return None

def _extract_gridcode_from_text(html_soup):
    """Extract gridcode from text format (fallback)."""
    text_content = html_soup.get_text(separator='|')
    parts = text_content.split('|')
    for i, part in enumerate(parts):
        if 'gridcode' in part.lower():
            if ':' in part:
                return part.split(':')[-1].strip()
            elif i + 1 < len(parts):
                return parts[i + 1].strip()
    return None

KML Boundary → GeoDataFrame

In [13]:
def load_boundary_polygon(kml_path):
    """
    Load boundary polygon from KML file using GeoPandas.
    Returns GeoDataFrame.
    """
    gdf = gpd.read_file(kml_path, driver='KML')
    gdf = gdf.to_crs("EPSG:4326")
    return gdf

def filter_points_within_boundary(df, boundary_gdf):
    """
    Spatially filter points inside boundary polygon.
    Returns GeoDataFrame.
    """
    gdf_points = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.x, df.y),
        crs="EPSG:4326"
    )

    boundary_polygon = boundary_gdf.union_all()

    filtered = gdf_points[gdf_points.within(boundary_polygon)]
    return filtered

In [14]:
# Paths
input_kmz = "../data/raw/flood/bc5_20251124_2200.kmz"
boundary_kml = "../data/raw/Hatyai.kml"

os.makedirs("../data/processed", exist_ok=True)

# Step 1: KMZ → DataFrame
df_flood = kmz_to_dataframe(input_kmz)

# Step 2: Load boundary
boundary_gdf = load_boundary_polygon(boundary_kml)

# Step 3: Filter
df_filtered = filter_points_within_boundary(df_flood, boundary_gdf)

# Step 4: Save
boundary_output_path = "../data/processed/hatyai_boundary.geojson"
boundary_gdf.to_file(boundary_output_path, driver="GeoJSON")

print(f"Boundary saved to {boundary_output_path}")

flood_output_path = "../data/processed/flood_filtered_20251124.csv"
df_filtered.drop(columns="geometry").to_csv(flood_output_path, index=False)

print(f"Saved {len(df_filtered)} filtered points to {flood_output_path}")

Boundary saved to ../data/processed/hatyai_boundary.geojson
Saved 651 filtered points to ../data/processed/flood_filtered_20251124.csv
